# 1.2.6 Version Control with Git

Calling Git from Python via `subprocess`. Create a temp repo, commit, branch, diff, stash, tag.

In [ ]:
import subprocess, shutil, tempfile, json, re
from pathlib import Path

def git(*args, cwd=None):
    r=subprocess.run(['git',*args],capture_output=True,text=True,cwd=cwd)
    if r.returncode!=0: raise RuntimeError(r.stderr.strip())
    return r.stdout.strip()

print(git('--version'))

In [ ]:
# Create temp repo + 3 commits
repo=Path(tempfile.mkdtemp(prefix='git_demo_'))
git('init',cwd=repo)
git('config','user.email','test@test.com',cwd=repo)
git('config','user.name','Test User',cwd=repo)
(repo/'model.py').write_text('class Model: pass\n')
git('add','.',cwd=repo); git('commit','-m','feat: add model',cwd=repo)
(repo/'metrics.json').write_text(json.dumps({'acc':0.85}))
git('add','.',cwd=repo); git('commit','-m','chore: save metrics',cwd=repo)
print(git('log','--oneline',cwd=repo))

In [ ]:
# Branch + diff
git('checkout','-b','experiment',cwd=repo)
(repo/'model.py').write_text('class Model:\n    lr=0.01\n')
print(git('diff','model.py',cwd=repo)[:400])
git('checkout','-',cwd=repo)

In [ ]:
# Stash
(repo/'wip.py').write_text('# wip\n')
git('add','wip.py',cwd=repo)
git('stash','push','-m','WIP',cwd=repo)
print('After stash — wip.py exists:', (repo/'wip.py').exists())
git('stash','pop',cwd=repo)
print('After pop   — wip.py exists:', (repo/'wip.py').exists())

In [ ]:
# Tag + cleanup
git('tag','-a','v1.0','-m','release v1.0',cwd=repo)
print('Tags:', git('tag',cwd=repo))
shutil.rmtree(repo); print('Cleaned up')